In [2]:
import os 
import sys
import pandas as pd 
import numpy as np 

import plotly.graph_objects as go 

OR_LEARNING_PATH = os.path.join(os.getcwd().split('OR_learning')[0], 'OR_learning/')
sys.path.insert(0, os.path.join(OR_LEARNING_PATH, 'utils/'))

import BindingCavity_functions as bc 
import SequenceAlignment_functions as sa
import plot_functions as pf
import color_function as cf 
import pdb_functions as pu
import voxel_functions as vf

In [3]:
import importlib 

importlib.reload(bc)
importlib.reload(pf)
importlib.reload(sa)
importlib.reload(pu)
importlib.reload(vf)
importlib.reload(cf)


<module 'color_function' from '/mnt/data2/Justice/OR_learning/utils/color_function.py'>

### AlphaFold3

Use pykvFinder and test whether defined cavity and identified residues includes the experimentally confirmed resiudes for Or51E2<br>

OR51E2 residues within 5A from propionate<br>
`H104, F155, L158, H180, Q181, N194, G198, L199, A201, I202, S258, R262`<br>
<br>
`R262` is ClassI-conserved residue crucial for carboxylic acid binding<br>
`F155` is known to be important for determining the size of carboxylic acid ligands<br>

Test how different AlphaFold3 model may be improvement for using structural analysis to identify binding cavity and cavity residues <br>
Things to test consists of: 
- Testing AF3 Or51E2 against different mutations of AF3 Or51E2 in the structural paper. This can potentially recapitulate the idea that binding cavity shape and interacting residues changes by mutation. And more importantly can be shown in via homology model 
- Conduct similar pipeline of comparing AF3 Or51E2 against cryo_OR51E2, but the HM should be built on consOR51 as a template. 

#### AF3 Or51E2 mutants

In [4]:
"""
Quickly convert all .cif in AF_out dir to .pdb 
"""

base_dir = "/mnt/data2/Justice/AF3_files/AF3_out/"
OVERWRITE = False

for sub_dir in os.listdir(base_dir):
    sub_path = os.path.join(base_dir, sub_dir)

    if os.path.isdir(sub_path):  # Ensure it's a directory
        cif_file       = os.path.join(sub_path, f"{sub_dir}_model.cif")
        pdb_file       = os.path.join(sub_path, f"{sub_dir}_model.pdb")
        nolig_pdb_file = os.path.join(sub_path, f"{sub_dir}_model_nolig.pdb")

        if os.path.exists(cif_file):  # Check if the .cif file exists
            if OVERWRITE or not os.path.exists(pdb_file):  # Check if the .pdb file is missing
                print(f"Converting {cif_file} to {pdb_file}...")
                pu.cif_to_pdb(cif_file, pdb_file)  # Convert
                pu.fix_pdb_format(pdb_file, pdb_file, 
                              chain_id = 'A') # Filter for chain A only, to exclude other complexes such as Ga
                pu.fix_pdb_format(pdb_file, nolig_pdb_file, keep_ligand=False)
            else:
                print(f"{pdb_file} already exists, skipping conversion.")
        else:
            print(f"No .cif file found in {sub_path}, skipping.")

/mnt/data2/Justice/AF3_files/AF3_out/consor51_ga_c9/consor51_ga_c9_model.pdb already exists, skipping conversion.
/mnt/data2/Justice/AF3_files/AF3_out/or51e2/or51e2_model.pdb already exists, skipping conversion.
No .cif file found in /mnt/data2/Justice/AF3_files/AF3_out/pS6_screen, skipping.
/mnt/data2/Justice/AF3_files/AF3_out/or51e1_i205a_ga_c9/or51e1_i205a_ga_c9_model.pdb already exists, skipping conversion.
/mnt/data2/Justice/AF3_files/AF3_out/or51e1_ga_c9/or51e1_ga_c9_model.pdb already exists, skipping conversion.
/mnt/data2/Justice/AF3_files/AF3_out/or51e1_m158a_ga_c9/or51e1_m158a_ga_c9_model.pdb already exists, skipping conversion.
Converting /mnt/data2/Justice/AF3_files/AF3_out/or51e1_ga/or51e1_ga_model.cif to /mnt/data2/Justice/AF3_files/AF3_out/or51e1_ga/or51e1_ga_model.pdb...
Converted PDB saved to: /mnt/data2/Justice/AF3_files/AF3_out/or51e1_ga/or51e1_ga_model.pdb
Fixed PDB saved to: /mnt/data2/Justice/AF3_files/AF3_out/or51e1_ga/or51e1_ga_model.pdb
Fixed PDB saved to: /mnt

In [ ]:
"""
Quick visualization of AF3 structure with ligand
"""

fig = pf.plot_coordinates([pu.load_pdb_coordinates(os.path.join(base_dir, 'consor51_ga_c9/consor51_ga_c9_model.pdb'), 
                                                  keep_ligand=True)[1], 
                           pu.load_pdb_coordinates(os.path.join(base_dir, 'consor51_ga_c9/consor51_ga_c9_model.pdb'), 
                                                  keep_ligand=True)[3]], 
                          colors = cf.distinct_colors([0,1], form='list')
                    )

fig.show()

In [6]:
"""
Align files 
"""

base_dir = "/mnt/data2/Justice/AF3_files/AF3_out/"

pdb_files = [os.path.join(base_dir, 'consor51_ga_c9/consor51_ga_c9_model.pdb'), 
             os.path.join(base_dir, 'or51e1_ga/or51e1_ga_model.pdb'),             
             os.path.join(base_dir, 'or51e1_ga_c9/or51e1_ga_c9_model.pdb'),
             os.path.join(base_dir, 'or51e1_i205a_ga_c9/or51e1_i205a_ga_c9_model.pdb'), 
             os.path.join(base_dir, 'or51e1_m158a_ga_c9/or51e1_m158a_ga_c9_model.pdb')]

for _pdb in pdb_files: # tmalign the pdb first
    pu.tmalign_pdb("/mnt/data2/Justice/AF3_files/AF3_out/consor51_ga_c9/consor51_ga_c9_model.pdb", 
                   _pdb, 
                   _pdb)
    pu.tmalign_pdb("/mnt/data2/Justice/AF3_files/AF3_out/consor51_ga_c9/consor51_ga_c9_model.pdb", 
                   _pdb.replace('.pdb', '_nolig.pdb'), 
                   _pdb.replace('.pdb', '_nolig.pdb'))

Aligned PDB saved to: /mnt/data2/Justice/AF3_files/AF3_out/consor51_ga_c9/consor51_ga_c9_model.pdb
Aligned PDB saved to: /mnt/data2/Justice/AF3_files/AF3_out/consor51_ga_c9/consor51_ga_c9_model_nolig.pdb
Aligned PDB saved to: /mnt/data2/Justice/AF3_files/AF3_out/or51e1_ga/or51e1_ga_model.pdb
Aligned PDB saved to: /mnt/data2/Justice/AF3_files/AF3_out/or51e1_ga/or51e1_ga_model_nolig.pdb
Aligned PDB saved to: /mnt/data2/Justice/AF3_files/AF3_out/or51e1_ga_c9/or51e1_ga_c9_model.pdb
Aligned PDB saved to: /mnt/data2/Justice/AF3_files/AF3_out/or51e1_ga_c9/or51e1_ga_c9_model_nolig.pdb
Aligned PDB saved to: /mnt/data2/Justice/AF3_files/AF3_out/or51e1_i205a_ga_c9/or51e1_i205a_ga_c9_model.pdb
Aligned PDB saved to: /mnt/data2/Justice/AF3_files/AF3_out/or51e1_i205a_ga_c9/or51e1_i205a_ga_c9_model_nolig.pdb
Aligned PDB saved to: /mnt/data2/Justice/AF3_files/AF3_out/or51e1_m158a_ga_c9/or51e1_m158a_ga_c9_model.pdb
Aligned PDB saved to: /mnt/data2/Justice/AF3_files/AF3_out/or51e1_m158a_ga_c9/or51e1_m158

In [7]:
"""
Filter the cavity and cavity res via defined canonical binding pocket zone to see what is preserved 
"""

OR_LEARNING_PATH = os.path.join(os.getcwd().split('OR_learning')[0], 'OR_learning/')
canonical_bc_coords = pd.read_pickle(os.path.join(OR_LEARNING_PATH,'files/binding_cavity/canonical_bc_coords.pkl'))

# Filtering ORs cavity and residue coordinates if they overlap with the defined canonical binding cavity
Cbc_cav_coords = { _Or: bc.filter_coordinates_within_cavity(canonical_bc_coords, 
                                                             np.array(cavsurf_coords[_Or])) for _Or in cavsurf_coords}

Cbc_res_coords = { _Or: bc.filter_coordinates_within_cavity(canonical_bc_coords, 
                                                             np.array(res_coords[_Or]), 
                                                             is_residue=True) for _Or in res_coords}


In [ ]:
pdb_files = [os.path.join(base_dir, 'consor51_ga_c9/consor51_ga_c9_model_nolig.pdb'), 
             os.path.join(base_dir, 'or51e1_ga/or51e1_ga_model_nolig.pdb'),             
             os.path.join(base_dir, 'or51e1_ga_c9/or51e1_ga_c9_model_nolig.pdb'),
             os.path.join(base_dir, 'or51e1_i205a_ga_c9/or51e1_i205a_ga_c9_model_nolig.pdb'), 
             os.path.join(base_dir, 'or51e1_m158a_ga_c9/or51e1_m158a_ga_c9_model_nolig.pdb')]

# Run pyKVFinder
parameter_set = {"probe_in": 1.0, "probe_out": 3.0, "removal_distance": 2.0, "volume_cutoff": 20.0}
cav_coords, cavsurf_coords, res_coords = bc.run_pyKVFinder_workflow(pdb_files, parameter_set=parameter_set)


# Add cavity surf 
plot_data = [cavsurf_coords[_key] for _key in cavsurf_coords]
labels = [_key + '_cav' for _key in cavsurf_coords.keys()]

# Plot
colormap = cf.distinct_colors(list(range(len(labels))), form='list')
fig = pf.plot_coordinates(plot_data, 
                          labels=labels, 
                          colors=colormap, 
                          opacity=0.1, 
                          size=5,
                          mode='markers')

# Add cavity contact residues 
# plot_data = [np.array(res_coords[_key])[:,3:6].astype(float) for _key in res_coords]
# fig_res= pf.plot_coordinates(plot_data, 
#                              labels=[_key + '_res' for _key in cavsurf_coords.keys()], 
#                              colors=colormap, 
#                              opacity=0.5, 
#                              size=5,
#                              mode='markers')
# background 
fig_backbone = pf.plot_coordinates([pu.load_pdb_coordinates("/mnt/data2/Justice/AF3_files/AF3_out/consor51_ga_c9/consor51_ga_c9_model.pdb")[1]], 
                          labels=['backbone'], 
                          colors='#D3D3D3', 
                          opacity=0.5, 
                          size=5,
                          mode='markers')
# ligand
lig_files = [os.path.join(base_dir, 'consor51_ga_c9/consor51_ga_c9_model.pdb'),
            #  os.path.join(base_dir, 'or51e1_ga/or51e1_ga_model.pdb'), 
             os.path.join(base_dir, 'or51e1_ga_c9/or51e1_ga_c9_model.pdb'),
             os.path.join(base_dir, 'or51e1_i205a_ga_c9/or51e1_i205a_ga_c9_model.pdb'), 
             os.path.join(base_dir, 'or51e1_m158a_ga_c9/or51e1_m158a_ga_c9_model.pdb')]
fig_propionate = pf.plot_coordinates([pu.load_pdb_coordinates(_pdb, keep_ligand=True)[3] for _pdb in lig_files], 
                                     labels=[os.path.basename(_pdb).split('_model')[0] for _pdb in lig_files ], 
                                     colors='purple', 
                                     opacity=1, 
                                     size=7,
                                     mode='markers')
# fig.add_traces(fig_res.data)
fig.add_traces(fig_backbone.data)
fig.add_traces(fig_propionate.data)
fig.show()

fig.write_html('/mnt/data2/Justice/OR_learning/output/Canonical_bc/AF3/OR51E1_mutants/OR51E1_comparisons.html')

### Trouble shooting repairing filter_coordinates_within_cavity 
Should filter for res and cav as a whole identity not as a separate coordinates 

In [ ]:
OR_LEARNING_PATH = os.path.join(os.getcwd().split('OR_learning')[0], 'OR_learning/')
canonical_bc_coords = pd.read_pickle(os.path.join(OR_LEARNING_PATH,'files/binding_cavity/canonical_bc_coords.pkl'))

Cbc_surf_coords = { _Or: bc.filter_coordinates_within_cavity(canonical_bc_coords, 
                                                            cavsurf_coords[_Or]) for _Or in cavsurf_coords}


In [ ]:
pdb_files = ['/mnt/data2/Justice/AF3_files/AF3_out/or51e1_ga/or51e1_ga_model_nolig_tmaligned.pdb']


# for _pdb in pdb_files: # tmalign the pdb first
#     pu.tmalign_pdb("/mnt/data2/Justice/AF_files/AF_tmaligned_pdb/Olfr1377_tmaligned.pdb", 
#                    _pdb, 
#                    _pdb.replace('.pdb', '_tmaligned.pdb'))

    
canonical_bc_coords = pd.read_pickle(os.path.join(OR_LEARNING_PATH,'files/binding_cavity/canonical_bc_coords.pkl'))

cav_coords, cavsurf_coords, res_coords = bc.run_pyKVFinder_workflow(['/mnt/data2/Justice/AF3_files/AF3_out/or51e1_ga/or51e1_ga_model_nolig_tmaligned.pdb'], 
                                                                    # parameter_set=parameter_set, 
                                                                    cavity_identity=True)

Cbc_results = {
    _Or: bc.filter_coordinates_within_cavity(
        canonical_bc_coords, 
        cavsurf_coords[_Or], 
        residue_coordinates=res_coords[_Or] if _Or in res_coords else None
    ) 
    for _Or in cavsurf_coords
}

# Unpack results into separate dictionaries
Cbc_surf_coords = { _Or: Cbc_results[_Or][0] for _Or in Cbc_results }
Cbc_res_coords = { _Or: Cbc_results[_Or][1] for _Or in Cbc_results }  


plot_data = [Cbc_surf_coords[_key] for _key in Cbc_surf_coords]
labels = [_key + '_cav' for _key in Cbc_surf_coords.keys()]

# Plot
colormap = cf.distinct_colors([1,2,3], form='list')
fig = pf.plot_coordinates(plot_data, 
                          labels=labels, 
                          colors=colormap, 
                          opacity=0.1, 
                          size=5,
                          mode='markers')

# Add canonical binding cavity cloud
fig_CBC = pf.plot_coordinates(canonical_bc_coords, 
                              colors = 'black', 
                              labels = 'CBC',
                              opacity = 0.01)

# Add cavity contact residues 
fig_res= pf.plot_coordinates([Cbc_res_coords[_key][:,3:] for _key in Cbc_res_coords], 
                             labels=[_key + '_res' for _key in Cbc_res_coords.keys()], 
                             colors=colormap, 
                             opacity=0.5, 
                             size=5,
                             mode='markers')
# background 
fig_backbone = pf.plot_coordinates([pu.load_pdb_coordinates("/mnt/data2/Justice/AF3_files/AF3_out/or51e1_ga/or51e1_ga_model_nolig_tmaligned.pdb")[1]], 
                          labels=['backbone'], 
                          colors='#D3D3D3', 
                          opacity=0.5, 
                          size=5,
                          mode='markers')

fig.add_traces(fig_res.data)
fig.add_traces(fig_CBC.data)
fig.add_traces(fig_backbone.data)
fig.show()


### pS6IP screen

Take in structures generated via AlphaFold3, selected based on positive or none interacting OR-ligand pairs via pS6IP. <br>

Idea is to hopefully visualize the differences in binding between positive OR-ligand pair and none interacting pairs.  

In [ ]:
"""
Quickly convert all .cif in AF_out dir to .pdb 
"""

align_ref_pdb = '/mnt/data2/Justice/AF_files/AF_tmaligned_pdb/Olfr1377_tmaligned.pdb'
base_dir = "/mnt/data2/Justice/AF3_files/AF3_out/pS6_screen/"
OVERWRITE = False

for sub_dir in os.listdir(base_dir):
    sub_path = os.path.join(base_dir, sub_dir)

    if os.path.isdir(sub_path):  # Ensure it's a directory
        cif_file       = os.path.join(sub_path, f"{sub_dir}_model.cif")
        pdb_file       = os.path.join(sub_path, f"{sub_dir}_model.pdb")
        nolig_pdb_file = os.path.join(sub_path, f"{sub_dir}_model_nolig.pdb")

        if os.path.exists(cif_file):  # Check if the .cif file exists
            if OVERWRITE or not os.path.exists(pdb_file):  # Check if the .pdb file is missing
                print(f"Converting {cif_file} to {pdb_file}...")
                pu.cif_to_pdb(cif_file, pdb_file)  # Convert
                pu.fix_pdb_format(pdb_file, pdb_file, 
                              chain_id = 'A') # Filter for chain A only, to exclude other complexes such as Ga
                pu.tmalign_pdb(align_ref_pdb, pdb_file, pdb_file) # align to ref pdb for comparison later 
                pu.fix_pdb_format(pdb_file, nolig_pdb_file, keep_ligand=False)
            else:
                print(f"{pdb_file} already exists, skipping conversion.")
        else:
            print(f"No .cif file found in {sub_path}, skipping.")

In [ ]:
"""
Read in pdb files, run pyKVFinder and filter for binding cavity

"""

# get all pdb file paths in pS6_screen 
base_dir = "/mnt/data2/Justice/AF3_files/AF3_out/pS6_screen/"
pdb_files = [os.path.join(base_dir, sub_dir, f"{sub_dir}_model_nolig.pdb") for sub_dir in os.listdir(base_dir)] 

# Run pyKVFinder and Filter 
cav_coords, cavsurf_coords, res_coords = bc.run_pyKVFinder_workflow(pdb_files,                                                             
                                                                    cavity_identity=True)



filter_cutoff = 0.8
canonical_bc_coords = pd.read_pickle(os.path.join(OR_LEARNING_PATH,'files/binding_cavity/canonical_bc_coords.pkl'))

Cbc_cav_coords = {
    _key.split('_')[0]+'_'+_key.split('_')[2]: bc.filter_coordinates_within_cavity(
        canonical_bc_coords, 
        cavity_coordinates  = cav_coords[_key], 
        residue_coordinates = res_coords[_key] if _key in res_coords else None, 
        filter_cutoff = filter_cutoff
        )[0]
    for _key in cav_coords
}


Cbc_results = {
    _Or: bc.filter_coordinates_within_cavity(
        canonical_bc_coords, 
        cavsurf_coords[_Or], 
        residue_coordinates=res_coords[_Or] if _Or in res_coords else None, 
        filter_cutoff = filter_cutoff
    ) 
    for _Or in cavsurf_coords
}
# Unpack results into separate dictionaries
Cbc_surf_coords = {_key.split('_')[0]+'_'+_key.split('_')[2]: Cbc_results[_key][0] for _key in Cbc_results}
Cbc_res_coords = {_key.split('_')[0]+'_'+_key.split('_')[2]: Cbc_results[_key][1] for _key in Cbc_results}


import pickle
with open('/mnt/data2/Justice/OR_learning/files/pS6_screen/dict_pS6_screen_CBC_cav_coords.pkl', 'wb') as path:
    pickle.dump(Cbc_cav_coords, path)
with open('/mnt/data2/Justice/OR_learning/files/pS6_screen/dict_pS6_screen_CBC_surf_coords.pkl', 'wb') as path:
    pickle.dump(Cbc_surf_coords, path)
with open('/mnt/data2/Justice/OR_learning/files/pS6_screen/dict_pS6_screen_CBC_res_coords.pkl', 'wb') as path:
    pickle.dump(Cbc_res_coords, path)


In [ ]:
pS6_OR = pd.read_csv('/mnt/data2/Justice/OR_learning/output/Canonical_bc/AF3/pS6_screen/or_screen.csv', 
                        index_col = 0)
pS6_OR['dl_or'] = pS6_OR['DL_OR'].str.lower()
pS6_OR['or_cid'] = pS6_OR['dl_or'] + '_' + pS6_OR['cid'].astype(str)

plot_data = [Cbc_cav_coords[_key] for _key in pS6_OR['or_cid'] if _key in Cbc_cav_coords.keys()]
labels = [_key for _key in list(pS6_OR['dl_or'] + '_' + pS6_OR['cid'].astype(str))]

# Plot
colormap = cf.distinct_colors(list(range(len(labels))), form='list')
fig = pf.plot_coordinates(plot_data, 
                          labels=labels, 
                          colors=colormap, 
                          opacity=0.1, 
                          size=5,
                          mode='markers')

# Add ligand coordinates  
ligand_coords = [pu.load_pdb_coordinates(_pdb.replace('_nolig.pdb', '.pdb'), keep_ligand=True)[3] \
    for _label in labels \
        for _pdb in pdb_files if '_'.join([_label.split('_')[0],'ga', _label.split('_')[1]]) in _pdb]
fig_lig = pf.plot_coordinates(ligand_coords, 
                              labels=labels, 
                              colors=colormap, 
                              opacity=0.8, 
                              size=7,
                              mode='markers')
# Assign legend groups to traces  
for trace in fig.data:
    trace.legendgroup = trace.name  # Group traces by the same name  
    trace.showlegend = True  # Show legend for one representative trace  

for trace in fig_lig.data:
    trace.legendgroup = trace.name  # Use the same group as cavity points  
    trace.showlegend = False  # Hide duplicate legends  

# Add ligand traces to the main figure  
fig.add_traces(fig_lig.data)

# Add canonical binding cavity cloud
fig_CBC = pf.plot_coordinates(canonical_bc_coords, 
                              colors = 'black', 
                              labels = ['CBC'],
                              opacity = 0.1)

# background 
fig_backbone = pf.plot_coordinates([pu.load_pdb_coordinates('/mnt/data2/Justice/AF_files/AF_tmaligned_pdb/Olfr1377_tmaligned.pdb')[1]], 
                          labels=['backbone'], 
                          colors='#D3D3D3', 
                          opacity=0.5, 
                          size=5,
                          mode='markers')

fig.add_traces(fig_CBC.data)
fig.add_traces(fig_backbone.data)
fig.show()

fig.write_html('/mnt/data2/Justice/OR_learning/output/Canonical_bc/AF3/pS6_screen/pS6_screen_model.html')

### AF3 Binding Pocket 

#TODO instead of expansion by 3A, don't expand use a threshold comparison to see the threshold to keep most the cavity.<br>
#TODO how much overlap, or num residue is between cav found residue and 5A residue <br>
#DONE grantham distance 5A res is a good place to start 

With the new AF3 OR-ligand structure. Create a new binding pocket filter cutoff. 

In [ ]:
"""
Read in pdb files, run pyKVFinder and filter for binding cavity

ONLY using the OR-ligand with FDR < 0.05 via pS6-IP experiments 

"""

# get all pdb file paths in pS6_screen 
base_dir = "/mnt/data2/Justice/AF3_files/AF3_out/pS6_screen/"
pdb_files = [os.path.join(base_dir, sub_dir, f"{sub_dir}_model_nolig.pdb") for sub_dir in os.listdir(base_dir)] 

# Filter pdb_files by only the ones with FDR < 0.05
pS6_OR = pd.read_csv('/mnt/data2/Justice/OR_learning/output/Canonical_bc/AF3/pS6_screen/or_screen.csv', index_col=0)
pos_or_cid = list(pS6_OR[pS6_OR.FDR < 0.05].or_cid.values)
pdb_files = [_pdb for _pdb in pdb_files if "_".join([os.path.basename(_pdb).split('_')[i] for i in (0, 2)]) in pos_or_cid]

# Run pyKVFinder and Filter 
cav_coords, cavsurf_coords, res_coords = bc.run_pyKVFinder_workflow(pdb_files,                                                             
                                                                    cavity_identity=True)

In [59]:
"""
Filter each AF3 cavity result by keeping only cavity closest to ligand 
"""

for _file in pdb_files: 
    or_key = os.path.basename(_file).replace('.pdb','')
    
    # Extract OR approximate cav and lig centers
    cav_center = {_cav: np.mean(cav_coords[or_key][_cav], axis=0) for _cav in cav_coords[or_key]}
    lig_center = np.mean(pu.load_pdb_coordinates(_file.replace('_nolig.pdb', '.pdb'), keep_ligand=True)[3], axis=0)
    
    # Get cav key closest to ligand
    dist_list = [np.linalg.norm(np.array(cav_center[_cav]) - np.array(lig_center)) for _cav in cav_center]
    cav_key = list(cav_center.keys())[np.argmin(dist_list)]
    
    # Filter result with cavity key 
    cav_coords[or_key] = cav_coords[or_key][cav_key]
    cavsurf_coords[or_key] = cavsurf_coords[or_key][cav_key]
    res_coords[or_key] = res_coords[or_key][cav_key]

In [ ]:
"""
Quick visualization of the cavity overlap. 

Including cryo structures
"""

plot_data = []
labels = []
for _file in pdb_files: 
    or_key = os.path.basename(_file).replace('.pdb','')
    # add keys for OR and ligand
    labels.append(or_key.replace('_model_nolig', ''))
    labels.append('_'.join([or_key.split('_')[0], or_key.split('_')[2]]))
    
    # or_data = [_cav for _cav in cavsurf_coords[or_key]]
    # or_data.extend(pu.load_pdb_coordinates(_file.replace('_nolig.pdb', '.pdb'), keep_ligand=True)[3])

    # plot_data.append(or_data)
    plot_data.append([_cav for _cav in cavsurf_coords[or_key]])
    plot_data.append(pu.load_pdb_coordinates(_file.replace('_nolig.pdb', '.pdb'), keep_ligand=True)[3])
    

# Add cryo Structure ligand
cryo_pdb = ['/mnt/data2/Justice/OR_learning/files/OR_seq/consOR1_tmaligned_nolig.pdb']
_, cryo_cavsurf_coords, _ = bc.run_pyKVFinder_workflow(cryo_pdb, cavity_identity=True)

cav_center = {_cav: np.mean(cryo_cavsurf_coords['consOR1_nolig'][_cav], axis=0) for _cav in cryo_cavsurf_coords['consOR1_nolig']}
lig_center = np.mean(pu.load_pdb_coordinates('/mnt/data2/Justice/OR_learning/files/OR_seq/consOR1_tmaligned.pdb', keep_ligand=True)[3], axis=0)
# Get cav key closest to ligand
dist_list = [np.linalg.norm(np.array(cav_center[_cav]) - np.array(lig_center)) for _cav in cav_center]
cryo_cavsurf_coords['consOR1_nolig'] = cryo_cavsurf_coords['consOR1_nolig'][list(cav_center.keys())[np.argmin(dist_list)]]

plot_data.append(cryo_cavsurf_coords['consOR1_nolig'])
plot_data.append(pu.load_pdb_coordinates('/mnt/data2/Justice/OR_learning/files/OR_seq/consOR1_tmaligned.pdb', keep_ligand=True)[3])


# Add backbone for visualization 
plot_data.append(pu.load_pdb_coordinates(_file.replace('_nolig.pdb', '.pdb'))[1])

fig = pf.plot_coordinates(plot_data, 
                    labels = labels + ['consOR1', 'consOR1_lmenthol', 'backbone'],
                    opacity = [0.1 if i%2 == 0 else 0.8 for i in range(len(plot_data))])

fig.show()
fig.write_html('/mnt/data2/Justice/OR_learning/output/Canonical_bc/AF3/define_AF3_CBC/pS6_screen_ligand_cav.html')

In [ ]:
"""
Super impose and define binding cavity zone using the overlapping cav from AF3 

Thresholding at at least 50% overlapping area between all the cav 
"""
# Getting expanded coords for defined ORs 
expanded_coords, largest_cavity_coords = bc.define_binding_cavity_zone(cavsurf_coords, 
                                                                       expansion_distance=3)

# Trim coordinates to speed up procees in making voxel 
expanded_coords_trimmerd = {}
for _Or in expanded_coords: 
    expanded_coords_trimmerd[_Or] = np.unique(expanded_coords[_Or] // 1, axis=0)
    
# Identify Canonical Binding Cavity by threshold intersection of expanded cavity coordinates 
threshold = 0.5
min_count = int(threshold * len(expanded_coords_trimmerd))
all_coordinates = np.vstack(list(expanded_coords_trimmerd.values()))
unique_coordinates, counts = np.unique(all_coordinates, axis=0, return_counts=True)

canonical_bc_coords = unique_coordinates[counts >= min_count]

import pickle 
with open('/mnt/data2/Justice/OR_learning/files/binding_cavity/canonical_bc_coords.pkl', 'wb') as f:
    pickle.dump(canonical_bc_coords, f)

In [65]:
"""
Quick visualization of the cavity overlap. 

Including cryo structures
"""

plot_data = []
labels = []

# Load and append various coordinates
plot_data.append(cavsurf_coords['or1e1c_ga_1049_model_nolig'])
plot_data.append(expanded_coords['or1e1c_ga_1049_model_nolig'])
plot_data.append(pu.load_pdb_coordinates('/mnt/data2/Justice/AF3_files/AF3_out/pS6_screen/or1e1c_ga_1049/or1e1c_ga_1049_model.pdb',
                                         keep_ligand=True)[3])
plot_data.append(canonical_bc_coords)

# Add cryo structure ligand
cryo_pdb = ['/mnt/data2/Justice/OR_learning/files/OR_seq/consOR1_tmaligned_nolig.pdb']
_, cryo_cavsurf_coords, _ = bc.run_pyKVFinder_workflow(cryo_pdb, cavity_identity=True)

# Compute cavity center coordinates
cav_center = {_cav: np.mean(cryo_cavsurf_coords['consOR1_nolig'][_cav], axis=0) for _cav in cryo_cavsurf_coords['consOR1_nolig']}
lig_center = np.mean(pu.load_pdb_coordinates('/mnt/data2/Justice/OR_learning/files/OR_seq/consOR1_tmaligned.pdb', keep_ligand=True)[3], axis=0)

# Select cavity closest to ligand
dist_list = [np.linalg.norm(np.array(cav_center[_cav]) - np.array(lig_center)) for _cav in cav_center]
closest_cav_key = list(cav_center.keys())[np.argmin(dist_list)]
cryo_cavsurf_coords['consOR1_nolig'] = cryo_cavsurf_coords['consOR1_nolig'][closest_cav_key]

# Append cryo structure elements
plot_data.append(cryo_cavsurf_coords['consOR1_nolig'])
plot_data.append(pu.load_pdb_coordinates('/mnt/data2/Justice/OR_learning/files/OR_seq/consOR1_tmaligned.pdb', keep_ligand=True)[3])

# Add backbone
fig_backbone = pf.plot_coordinates(pu.load_pdb_coordinates(_file.replace('_nolig.pdb', '.pdb'))[1], 
                                   labels=['backbone'], opacity=0.5, colors=['#D3D3D3'])

# Create main figure
fig = pf.plot_coordinates(plot_data,
                          labels=['or1e1c_cavity', 'or1e1c_expanded_cavity', 'or1e1c_1049', 'filtered_cavity',
                                  'consOR1', 'consOR1_lmenthol'],
                          opacity=[0.1, 0.1, 0.8, 0.1, 0.1, 0.8])

# Add backbone trace
fig.add_traces(fig_backbone.data)

# Customize layout
fig.update_layout(title_text='Demo of OR binding cavity zone selection.<br>OR used to generate are from pS6-IP < 0.05 OR-ligand AF3 structures',
                  margin=dict(t=50))

# Show and save figure
fig.show()
fig.write_html('/mnt/data2/Justice/OR_learning/output/Canonical_bc/AF3/define_AF3_CBC/Demo_OR_cav_res_CBC.html')

OMP: Info #277: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
OMP: Info #277: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
OMP: Info #277: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
OMP: Info #277: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
OMP: Info #277: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
OMP: Info #277: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
OMP: Info #277: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
OMP: Info #277: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
OMP: Info #277: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
OMP: Info #277: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
OMP: Info #277: omp_set_nested

### PCA

In [25]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

Cbc_cav_coords  = pd.read_pickle('/mnt/data2/Justice/OR_learning/files/pS6_screen/dict_pS6_screen_CBC_cav_coords.pkl')
Cbc_surf_coords = pd.read_pickle('/mnt/data2/Justice/OR_learning/files/pS6_screen/dict_pS6_screen_CBC_surf_coords.pkl')
Cbc_res_coords  = pd.read_pickle('/mnt/data2/Justice/OR_learning/files/pS6_screen/dict_pS6_screen_CBC_res_coords.pkl')

In [ ]:
"""
Visualization of PCA input pre-voxelize
"""

plot_data = []
labels = []


plot_data.append(Cbc_res_coords['or1e1c_1049'][:,3:].astype(float))
plot_data.append(Cbc_surf_coords['or1e1c_1049'])
plot_data.append(pu.load_pdb_coordinates('/mnt/data2/Justice/AF3_files/AF3_out/pS6_screen/or1e1c_ga_1049/or1e1c_ga_1049_model.pdb',
                                         keep_ligand=True)[3])

fig = pf.plot_coordinates(plot_data,
                    labels =  ['or1e1c_residue', 'or1e1c_cav', 'or1e1c_1049'],
                    opacity = [0.5, 0.1, 0.8] 
                    )
fig_backbone = pf.plot_coordinates(pu.load_pdb_coordinates('/mnt/data2/Justice/AF3_files/AF3_out/pS6_screen/or1e1c_ga_1049/or1e1c_ga_1049_model.pdb')[1],
                          labels = ['backbone'], 
                          colors = ['#D3D3D3'],
                          opacity = 0.5)
fig.add_traces(fig_backbone.data)


CBC_coords = pd.read_pickle('/mnt/data2/Justice/OR_learning/files/binding_cavity/canonical_bc_coords.pkl')
fig_CBC = pf.plot_coordinates(CBC_coords, 
                    labels = ['CBC'],
                    colors = ['Purple'],
                    opacity = 0.1)
fig.add_traces(fig_CBC.data)

fig.update_layout(title_text = 'Residue and Cavity selected via filter of CBC', 
                  margin=dict(t=50))

fig.show()
# fig.write_html('/mnt/data2/Justice/OR_learning/output/Canonical_bc/AF3/define_AF3_CBC/Demo_OR_cav_res_CBC.html')

In [36]:
"""
Visualization of PCA input voxels 
"""

import plotly.graph_objects as go 

# Define a colormap for voxel properties (0-6)
color_map = {
    0: 'gray',       # Empty space
    1: 'blue',       # Aliphatic apolar
    2: 'purple',     # Aromatic
    3: 'green',      # Polar uncharged
    4: 'red',        # Negatively charged
    5: 'orange',     # Positively charged
    6: 'black'       # Non-standard residues
}

vis_index = [i for i, s in enumerate(Cbc_res_coords.keys()) if 'or1e1c_1049' in s.lower()]


voxelized_array, voxel_shape  = vf.voxelize_cavity(cavity_coords  = Cbc_surf_coords, 
                                                   residue_coords = Cbc_res_coords, 
                                                   resolution=1)

labeled_voxels = [vf.convert_properties(_voxels) for _voxels in voxelized_array]
voxel_data = np.array(labeled_voxels)[vis_index]
labels = [list(Cbc_res_coords.keys())[i] for i in vis_index]

fig = go.Figure()
for i, (label, voxel_grid) in enumerate(zip(labels, voxel_data)):
    for _property in np.unique(voxel_grid): 
        if _property == -1:
            continue  # Skip empty space
        
        # Get occupied voxels for the current property
        occupied_voxels = np.array(np.where(voxel_grid == _property)).T
        if len(occupied_voxels) == 0:
            continue  # Skip if no voxels found

        # Extract x, y, z coordinates
        x, y, z = occupied_voxels[:, 0], occupied_voxels[:, 1], occupied_voxels[:, 2]

        # Add scatter plot for the current property type
        fig.add_trace(go.Scatter3d(
            x=x, y=y, z=z,
            mode='markers',
            name=f"{label} - {str(_property)}",
            marker=dict(
                size=5,
                color=color_map[_property],  # Assign color based on property
                opacity=0.5
            ),
            # legend = label, 
            legendgroup = label
        ))
        
fig = pf._plotly_blank_style(fig)
fig.show()
# fig.write_html('/mnt/data2/Justice/OR_learning/output/Canonical_bc/AF3/define_AF3_CBC/Demo_vox_property.html')

In [ ]:
properties = vf.encode_residues_for_voxel(list(Cbc_res_coords.values()))[0][0][3]
print(properties)
print(extract_label_properties(properties))
print(properties[1:4])
print(properties[4:7])
print(properties[7:11])

[0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0]
-1
[1, 0, 0]
[0, 0, 0]
[1, 0, 0, 0]


In [79]:
print(properties[0][3])
print(properties[20][3])
print(properties[40][3])

[0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0]
[0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0]
[0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0]


In [80]:
properties = vf.encode_residues_for_voxel(list(Cbc_res_coords.values()))[0]
print(extract_label_properties(properties[0][3]))
print(extract_label_properties(properties[20][3]))
print(extract_label_properties(properties[40][3]))


-1
-4096
-65263


In [54]:
# Combine key discriminative properties
hydrophobicity = np.argmax(properties[1:4]) if np.any(properties[1:4]) else -1
charge = np.argmax(properties[4:7]) if np.any(properties[4:7]) else -1
h_bond = np.argmax(properties[7:11]) if np.any(properties[7:11]) else -1
aromaticity = np.argmax(properties[11:13]) if np.any(properties[11:13]) else -1
size_flex = np.argmax(properties[13:18]) if np.any(properties[13:18]) else -1

# Create a composite label that captures multiple properties
# Use bit-shifting to create a unique identifier
composite_label = (
    (hydrophobicity << 16) | 
    (charge << 12) | 
    (h_bond << 8) | 
    (aromaticity << 4) | 
    size_flex
)

In [ ]:
"""
PCA of AF3 voxelized cavity and residue properties 
"""
# Voxelize and pca 
pca_df, variance_ratio = vf.coords_to_voxel_pca(cavity_coords  = Cbc_cav_coords,
                                                residue_coords = Cbc_res_coords,
                                                resolution = 1 )

pS6_OR = pd.read_csv('/mnt/data2/Justice/OR_learning/output/Canonical_bc/AF3/pS6_screen/or_screen.csv', 
                        index_col = 0)
# Merge pS6 information to pca_df 
plot_data = pd.merge(pca_df,pS6_OR, on='or_cid')
plot_data['cid'] = plot_data['cid'].astype(str)


colormap = cf.distinct_colors(plot_data['DL_OR'].unique(), category='tab20')
shape_dict = cf.distinct_shapes(plot_data['cid'].unique())

fig = go.Figure()
for (_OR, _cid), subset in plot_data.groupby(["DL_OR", "cid"]):
    trace_name = f"{_OR}_{_cid}"  # Unique name for each trace
    fig.add_trace(go.Scatter(
        x=subset['PCA_1'],
        y=subset['PCA_2'],
        name=trace_name,  # Unique trace for each (DL_OR, cid)
        marker=dict(size    = 15 if subset.FDR.values[0] < 0.05 else 5, 
                    color   = colormap[_OR], 
                    symbol  = shape_dict[_cid], 
                    opacity = 0.7),
        hovertemplate=f'{_OR}<br>{subset.odor.values[0]}<br>logFC: {subset.logFC.values[0]:.2f}<br>FDR: {subset.FDR.values[0]:.2e}', 
        mode='markers',
        legendgroup= _OR, 
        showlegend=True
    ))

fig.update_layout(template   = 'simple_white', 
                  title_text = 'PCA of AF3 voxelized cavity and residue properties')
fig.update_xaxes(title_text=f"PCA_1 ({100*variance_ratio[0]:.3f}%)")
fig.update_yaxes(title_text=f"PCA_2 ({100*variance_ratio[1]:.3f}%)")

fig.show()
fig.write_html('/mnt/data2/Justice/OR_learning/output/Canonical_bc/AF3/pS6_screen/PCA/PCA_pS6_screen_cav_res.html')

Original features: 16224, Reduced features: 2046
Reduced data shape: (73, 2)
Explained variance ratio: [0.07377055 0.06892427]


In [ ]:
"""
PCA of AF3 voxelized residue properties 
"""
# Voxelize and pca 
pca_df, variance_ratio = vf.coords_to_voxel_pca(residue_coords = Cbc_res_coords,
                                                resolution = 1 )

pS6_OR = pd.read_csv('/mnt/data2/Justice/OR_learning/output/Canonical_bc/AF3/pS6_screen/or_screen.csv', 
                        index_col = 0)
# Merge pS6 information to pca_df 
plot_data = pd.merge(pca_df,pS6_OR, on='or_cid')
plot_data['cid'] = plot_data['cid'].astype(str)


colormap = cf.distinct_colors(plot_data['DL_OR'].unique(), category='tab20')
shape_dict = cf.distinct_shapes(plot_data['cid'].unique())

fig = go.Figure()
for (_OR, _cid), subset in plot_data.groupby(["DL_OR", "cid"]):
    trace_name = f"{_OR}_{_cid}"  # Unique name for each trace
    fig.add_trace(go.Scatter(
        x=subset['PCA_1'],
        y=subset['PCA_2'],
        name=trace_name,  # Unique trace for each (DL_OR, cid)
        marker=dict(size    = 15 if subset.FDR.values[0] < 0.05 else 5, 
                    color   = colormap[_OR], 
                    symbol  = shape_dict[_cid], 
                    opacity = 0.7),
        hovertemplate=f'{_OR}<br>{subset.odor.values[0]}<br>logFC: {subset.logFC.values[0]:.2f}<br>FDR: {subset.FDR.values[0]:.2e}', 
        mode='markers',
        legendgroup= _OR, 
        showlegend=True
    ))

fig.update_layout(template   = 'simple_white', 
                  title_text = 'PCA of AF3 voxelized residue properties')
fig.update_xaxes(title_text=f"PCA_1 ({100*variance_ratio[0]:.3f}%)")
fig.update_yaxes(title_text=f"PCA_2 ({100*variance_ratio[1]:.3f}%)")

fig.show()
fig.write_html('/mnt/data2/Justice/OR_learning/output/Canonical_bc/AF3/pS6_screen/PCA/PCA_pS6_screen_res.html')

Original features: 16224, Reduced features: 1521
Reduced data shape: (73, 2)
Explained variance ratio: [0.06438308 0.060572  ]


### Spatial encoded PCA

In [3]:
import importlib 

importlib.reload(bc)
importlib.reload(pf)
importlib.reload(sa)
importlib.reload(pu)
importlib.reload(cf)
importlib.reload(vf)


<module 'voxel_functions' from '/mnt/data2/Justice/OR_learning/utils/voxel_functions.py'>

In [4]:
Cbc_cav_coords  = pd.read_pickle('/mnt/data2/Justice/OR_learning/files/pS6_screen/dict_pS6_screen_CBC_cav_coords.pkl')
Cbc_res_coords  = pd.read_pickle('/mnt/data2/Justice/OR_learning/files/pS6_screen/dict_pS6_screen_CBC_res_coords.pkl')

In [ ]:
"""
TESTING OF DIFFERENT SPATIAL PCA . . . 
"""

voxelized_4D_arrays, voxel_shape = vf.voxelize_cavity(
    # cavity_coords = list(Cbc_cav_coords.values()),
    residue_coords = list(Cbc_res_coords.values()),
    resolution = 1 )
voxelized_3Dlabeled_arrays = np.array([vf.convert_properties(voxel) for voxel in voxelized_4D_arrays])

# Compare all methods to find which works best for your data
results = compare_encoding_methods(voxelized_4D_arrays, use_labeled=False)


Applying kmer encoding...
Applying weighted encoding...
Applying fourier encoding...
Applying spherical encoding...


In [ ]:
list(Cbc_res_coords.values())[0]

In [6]:
temp, _ = vf.voxelize_cavity(
    residue_coords = list(Cbc_res_coords.values()),
    resolution = 1 )

temp2 = np.array([vf.convert_properties(voxel) for voxel in temp])

In [200]:
print(temp[0][28][13][10:12])
print(property_preserving_smoothing(temp2[0], sigma=0.5, use_labeled=True)[28][13][10:12])

[[0 0 0 0 0 0 0]
 [0 1 0 0 0 0 0]]
[[0.0000000e+00 7.4798726e-02 0.0000000e+00 0.0000000e+00 0.0000000e+00
  0.0000000e+00 0.0000000e+00]
 [0.0000000e+00 4.8804381e-01 0.0000000e+00 2.2093704e-05 0.0000000e+00
  0.0000000e+00 0.0000000e+00]]


In [ ]:
property_preserving_smoothing(temp2[0], sigma=0.13, use_labeled=True)

In [28]:
print(temp[0][7][14][7].astype(float))
print(list(gaussian_blur_properties(temp[0], sigma=0.2))[7][14][7])

[0. 0. 0. 0. 1. 1. 0. 0. 1. 0. 0. 0. 1. 0. 1. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 1. 1. 0. 0. 1. 0. 0. 0. 1. 0. 1. 0. 0. 0. 0. 0. 0.]


In [16]:
# fig = pf.visualize_voxel_grid([temp[0], property_preserving_smoothing(temp2[0], sigma=1, use_labeled=True)], 
#                         voxel_type = '4D', 
#                         labels = ['test1', 'test2'], 
#                         opacity = 0.3,
#                         property_indices=[ 1,  2,  3,  4,  5]
#                         )
fig = pf.visualize_voxel_grid([temp[0],
                         gaussian_blur_properties(temp[0], sigma=0.2)], 
                        voxel_type = '4D', 
                        labels = ['original', 'gaussian_blurred'], 
                        opacity = 0.5,
                        property_indices=[ 1,2,3,4],
                        colormap=cf.distinct_colors(num_colors=10)
                        )
# fig.add_traces(fig_blurr.data)
fig.show()
# fig.write_html('/mnt/data2/Justice/OR_learning/output/Canonical_bc/AF3/pS6_screen/misc/Gaussian_blurr_voxel.html')

In [357]:
data = []
data.append(fourier_transform_voxel(temp[0], freq_cutoff=5))
data.append(fourier_transform_voxel(gaussian_blur_properties(temp[0], sigma=0.12), freq_cutoff=5))
data.append(fourier_transform_voxel(gaussian_blur_properties(temp[0], sigma=1), freq_cutoff=5))
# data.append(fourier_transform_voxel(temp[-3], freq_cutoff=5))
# data.append(fourier_transform_voxel(gaussian_blur_properties(temp[-3], sigma=1), freq_cutoff=5))

names = ['temp1', 'temp1_blurred', 'temp2', 'temp2_blurred']

fig = go.Figure()
for i in range(len(data)):
    fig.add_traces(
        go.Scatter(x = list(range(len(data[0]))),
                   y = data[i], 
                   name = names[i]
                #    marker=dict(opacity=0.5)
                )
    )
fig.update_traces(opacity=0.5)
fig.show()

In [22]:
importlib.reload(bc)
importlib.reload(pf)
importlib.reload(sa)
importlib.reload(pu)
importlib.reload(cf)
importlib.reload(vf)


<module 'voxel_functions' from '/mnt/data2/Justice/OR_learning/utils/voxel_functions.py'>

In [23]:
voxelized_array, voxel_shape = vf.voxelize_cavity(
    residue_coords=Cbc_res_coords,
    resolution=1
)

# Convert voxel properties
# labeled_voxels = np.array([vf.convert_properties(voxel) for voxel in voxelized_array])

In [8]:
import numpy as np
from scipy.ndimage import gaussian_filter
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

def gaussian_blur_properties(voxel_data, sigma=0.8, preserve_originals=True):
    """
    Apply Gaussian blur to each property channel separately without intermixing properties.
    
    Parameters:
    -----------
    voxel_data : 4D array (x, y, z, properties)
        One-hot encoded voxel data
    sigma : float or sequence of floats
        Standard deviation for Gaussian kernel
    preserve_originals : bool
        If True, original non-zero values remain unchanged
        
    Returns:
    --------
    blurred_voxel : 4D array
        Blurred voxel with the same shape as input
    """
    x_size, y_size, z_size, n_properties = voxel_data.shape
    blurred_voxel = np.zeros_like(voxel_data, dtype=np.float32)
    
    # Process each property channel separately
    for prop_idx in range(n_properties):
        prop_data = voxel_data[:, :, :, prop_idx]
        
        # Find original non-zero positions if we need to preserve them
        if preserve_originals:
            original_positions = prop_data > 0
        
        # Apply Gaussian blur to this property
        blurred_prop = gaussian_filter(prop_data.astype(np.float32), sigma=sigma)
        
        # If requested, preserve original values
        if preserve_originals:
            blurred_prop[original_positions] = prop_data[original_positions]
        
        # Store the blurred property
        blurred_voxel[:, :, :, prop_idx] = blurred_prop
    
    return blurred_voxel

def fourier_transform_voxel(voxel_data, freq_cutoff=3):
    """
    Apply 3D Fourier transform to extract spatial patterns from voxel data.
    
    Parameters:
    -----------
    voxel_data : 4D array (x, y, z, properties)
        Voxel data (can be blurred)
    freq_cutoff : int
        Maximum frequency to retain in the Fourier transform
        
    Returns:
    --------
    features : array
        Fourier-encoded features
    """
    x_size, y_size, z_size, n_properties = voxel_data.shape
    features = []
    
    # Apply 3D FFT to each property channel
    for prop in range(n_properties):
        prop_data = voxel_data[:, :, :, prop]
        
        # Compute 3D FFT
        fft_data = np.fft.fftn(prop_data)
        
        # Shift zero frequency to center
        fft_shifted = np.fft.fftshift(fft_data)
        
        # Get center indices
        center_x, center_y, center_z = x_size // 2, y_size // 2, z_size // 2
        
        # Extract magnitudes of low frequency components
        # These capture the overall spatial patterns
        for i in range(-freq_cutoff, freq_cutoff + 1):
            for j in range(-freq_cutoff, freq_cutoff + 1):
                for k in range(-freq_cutoff, freq_cutoff + 1):
                    if abs(i) + abs(j) + abs(k) <= freq_cutoff:  # Focus on lower frequencies
                        # Get FFT magnitude at this frequency
                        freq_i = center_x + i
                        freq_j = center_y + j
                        freq_k = center_z + k
                        
                        # Make sure the indices are within bounds
                        if (0 <= freq_i < x_size and 
                            0 <= freq_j < y_size and 
                            0 <= freq_k < z_size):
                            features.append(np.abs(fft_shifted[freq_i, freq_j, freq_k]))
    
    return np.array(features)

def blur_fourier_encode_voxels(voxels, sigma=0.8, preserve_originals=True, freq_cutoff=3):
    """
    Complete pipeline: blur, Fourier transform, and encoding for voxel data.
    
    Parameters:
    -----------
    voxels : list of 4D arrays
        List of 4D voxel data arrays
    sigma : float
        Sigma parameter for Gaussian blur
    preserve_originals : bool
        Whether to preserve original non-zero values
    freq_cutoff : int
        Maximum frequency to include in Fourier features
        
    Returns:
    --------
    encoded_features : 2D array
        Encoded features for each voxel
    """
    encoded_features = []
    
    for voxel in voxels:
        # Apply Gaussian blur to each property channel
        blurred_voxel = gaussian_blur_properties(voxel, sigma=sigma, preserve_originals=preserve_originals)
        
        # Extract Fourier features
        features = fourier_transform_voxel(blurred_voxel, freq_cutoff=freq_cutoff)
        
        encoded_features.append(features)
    
    return np.array(encoded_features)

def visualize_blurred_voxel(original_voxel, blurred_voxel, property_idx=1, slice_idx=None):
    """
    Visualize the effect of blurring on a specific property and slice.
    
    Parameters:
    -----------
    original_voxel : 4D array
        Original voxel data
    blurred_voxel : 4D array
        Blurred voxel data
    property_idx : int
        Index of the property to visualize
    slice_idx : int or None
        Index of the slice to visualize (if None, use middle slice)
    """
    x_size, y_size, z_size, _ = original_voxel.shape
    
    if slice_idx is None:
        slice_idx = z_size // 2
    
    # Extract the specified property and slice
    original_slice = original_voxel[:, :, slice_idx, property_idx]
    blurred_slice = blurred_voxel[:, :, slice_idx, property_idx]
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    
    im1 = ax1.imshow(original_slice, cmap='viridis')
    ax1.set_title(f'Original - Property {property_idx}, Slice {slice_idx}')
    plt.colorbar(im1, ax=ax1)
    
    im2 = ax2.imshow(blurred_slice, cmap='viridis')
    ax2.set_title(f'Blurred - Property {property_idx}, Slice {slice_idx}')
    plt.colorbar(im2, ax=ax2)
    
    plt.tight_layout()
    # return 

def fourier_pca_analysis(voxels, sigma=0.8, preserve_originals=True, freq_cutoff=3, n_components=2):
    """
    Apply the full pipeline: blur, Fourier transform, and PCA to voxel data.
    
    Parameters:
    -----------
    voxels : list of 4D arrays
        List of 4D voxel data arrays
    sigma : float
        Sigma parameter for Gaussian blur
    preserve_originals : bool
        Whether to preserve original non-zero values
    freq_cutoff : int
        Maximum frequency to include in Fourier features
    n_components : int
        Number of PCA components to extract
        
    Returns:
    --------
    transformed_data : 2D array
        PCA-transformed data
    pca : PCA object
        Fitted PCA model
    encoded_features : 2D array
        Encoded features for each voxel
    """
    # Apply blur and Fourier encoding
    encoded_features = blur_fourier_encode_voxels(
        voxels, sigma=sigma, preserve_originals=preserve_originals, freq_cutoff=freq_cutoff
    )
    
    # Apply PCA
    pca = PCA(n_components=n_components)
    transformed_data = pca.fit_transform(encoded_features)
    
    return transformed_data, pca, encoded_features

def evaluate_sigma_values(voxels, sigmas=[0.5, 0.8, 1.0, 1.5, 2.0], n_components=2):
    """
    Evaluate different sigma values for the blur step.
    
    Parameters:
    -----------
    voxels : list of 4D arrays
        List of 4D voxel data arrays
    sigmas : list of float
        Sigma values to evaluate
    n_components : int
        Number of PCA components to extract
        
    Returns:
    --------
    results : dict
        Results for each sigma value
    """
    results = {}
    
    plt.figure(figsize=(15, 10))
    
    for i, sigma in enumerate(sigmas):
        # Apply pipeline
        transformed_data, pca, _ = fourier_pca_analysis(
            voxels, sigma=sigma, n_components=n_components
        )
        
        # Store results
        results[sigma] = {
            'transformed_data': transformed_data,
            'explained_variance': pca.explained_variance_ratio_
        }
        
        # Plot
        plt.subplot(2, 3, i+1)
        plt.scatter(transformed_data[:, 0], transformed_data[:, 1], alpha=0.7)
        plt.title(f'Sigma = {sigma}\nVar: {pca.explained_variance_ratio_[0]:.2f}, {pca.explained_variance_ratio_[1]:.2f}')
        plt.xlabel('PC1')
        plt.ylabel('PC2')
        plt.grid(True, linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    # plt.savefig('sigma_comparison.png')
    # plt.close()
    
    return results

# Example pipeline for demonstration
def example_usage():
    # Create example 4D voxel data
    voxel_shape = (10, 10, 10, 7)  # x, y, z, properties (7 properties as you mentioned)
    n_samples = 5
    
    # Generate synthetic voxels for demonstration
    np.random.seed(42)
    voxels = []
    for _ in range(n_samples):
        voxel = np.zeros(voxel_shape)
        
        # Add some random active sites for each property
        for prop in range(1, 7):  # Properties 1-6 (skipping 0)
            num_sites = np.random.randint(3, 10)
            for _ in range(num_sites):
                x, y, z = np.random.randint(0, 10, size=3)
                voxel[x, y, z, prop] = 1
        
        voxels.append(voxel)
    
    # Demonstrate the blur effect
    voxel = voxels[0]
    blurred_voxel = gaussian_blur_properties(voxel, sigma=0.8)
    
    # Visualize for the first property and middle slice
    visualize_blurred_voxel(voxel, blurred_voxel, property_idx=1)
    
    # Complete pipeline with PCA
    transformed_data, pca, encoded_features = fourier_pca_analysis(voxels)
    
    print(f"Shape of encoded features: {encoded_features.shape}")
    print(f"Shape of transformed data: {transformed_data.shape}")
    print(f"Explained variance ratios: {pca.explained_variance_ratio_}")
    
    # Evaluate different sigma values
    results = evaluate_sigma_values(voxels)
    
    return transformed_data, encoded_features, results

In [80]:
from scipy.ndimage import gaussian_filter

print(temp2[0][11][25])
print(gaussian_filter(temp2[0], sigma=1)[11][25])

[-1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1  2  2  2 -1 -1 -1 -1 -1
 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1]
[-1  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0 -1 -1 -1 -1 -1 -1 -1]


In [ ]:

pf.visualize_voxel_grid([temp[0]], 
                        labels = ['original', 'uniform_filtered'], 
                        colormap = ['blue', 'green']
                        )

In [ ]:
fig = go.Figure()

fig.add_traces(
    go.Scatter3d(
        x = temp[:,0],
        y = temp[:,1],
        z = temp[:,2],
    )
)
fig.show()

In [155]:
from plotly.subplots import make_subplots


# Visualize the results
pS6_OR = pd.read_csv('/mnt/data2/Justice/OR_learning/output/Canonical_bc/AF3/pS6_screen/or_screen.csv', 
                        index_col = 0)

    
# Create a 2x2 subplot figure
fig = make_subplots(rows=2, cols=2, subplot_titles=list(results.keys()))
row_idx = 1
col_idx = 1
for _method in results:
    # Merge pS6 information to pca_df 
    pca_df = pd.DataFrame(results[_method]['transformed_data'], 
                          columns=[f'PCA_{i+1}' for i in range(2)], index=list(Cbc_res_coords.keys()))
    pca_df = pca_df.reset_index().rename(columns={'index': 'or_cid'})

    plot_data = pd.merge(pca_df, pS6_OR, on='or_cid')
    plot_data['cid'] = plot_data['cid'].astype(str)

    colormap = cf.distinct_colors(plot_data['DL_OR'].unique(), category='tab20')
    shape_dict = {_cid:['circle', 'square', 'triangle-up', 'x', 'diamond'][i] for i, _cid in enumerate(plot_data['cid'].unique())}

    # Loop through groups to add traces to the subplot
    for (_OR, _cid), subset in plot_data.groupby(["DL_OR", "cid"]):
        trace_name = f"{_OR}_{_cid}"  # Unique name for each trace
        fig.add_trace(
            go.Scatter(
                x=subset['PCA_1'],
                y=subset['PCA_2'],
                name=trace_name,
                marker=dict(
                    color=colormap[_OR], 
                    size=15 if subset.FDR.values[0] < 0.05 else 5, 
                    symbol=shape_dict[_cid], 
                    opacity=0.7
                ),
                hovertemplate=f'{_OR}<br>{subset.odor.values[0]}<br>logFC: {subset.logFC.values[0]:.2f}<br>FDR: {subset.FDR.values[0]:.2e}',
                mode='markers',
                legendgroup=_OR,
                # Only show in legend for first subplot to avoid duplicates
                showlegend=(row_idx == 1 and col_idx == 1)
            ),
            row=row_idx, col=col_idx
        )
    
    # Update axes labels for each subplot
    fig.update_xaxes(title_text=f"PCA_1 ({100*results[_method]['explained_variance'][0]:.3f}%)", row=row_idx, col=col_idx)
    fig.update_yaxes(title_text=f"PCA_2 ({100*results[_method]['explained_variance'][1]:.3f}%)", row=row_idx, col=col_idx)
    
    # Move to next subplot position
    col_idx += 1
    if col_idx > 2:
        col_idx = 1
        row_idx += 1

# Update layout for the entire figure
fig.update_layout(
    # height=800,  # Adjust height as needed
    # width=1000,  # Adjust width as needed
    template='simple_white',
    title_text="PCA Plots AF3 structure encoding method Comparison",
    legend_title_text="DL_OR_cid"
)

fig.show()
# fig.write_html('/mnt/data2/Justice/OR_learning/output/Canonical_bc/AF3/pS6_screen/PCA_encoding_method_cav_res.html')
fig.write_html('/mnt/data2/Justice/OR_learning/output/Canonical_bc/AF3/pS6_screen/PCA_encoding_method_res.html')

In [ ]:
from plotly.subplots import make_subplots


# Visualize the results
pS6_OR = pd.read_csv('/mnt/data2/Justice/OR_learning/output/Canonical_bc/AF3/pS6_screen/or_screen.csv', 
                        index_col = 0)

    
# Create a 2x2 subplot figure
fig = make_subplots(rows=2, cols=2, subplot_titles=list(results.keys()))
row_idx = 1
col_idx = 1
for _method in results:
    # Merge pS6 information to pca_df 
    pca_df = pd.DataFrame(results[_method]['transformed_data'], 
                          columns=[f'PCA_{i+1}' for i in range(2)], index=list(Cbc_res_coords.keys()))
    pca_df = pca_df.reset_index().rename(columns={'index': 'or_cid'})

    plot_data = pd.merge(pca_df, pS6_OR, on='or_cid')
    plot_data['cid'] = plot_data['cid'].astype(str)

    colormap = cf.distinct_colors(plot_data['DL_OR'].unique(), category='tab20')
    shape_dict = {_cid:['circle', 'square', 'triangle-up', 'x', 'diamond'][i] for i, _cid in enumerate(plot_data['cid'].unique())}

    # Loop through groups to add traces to the subplot
    for (_OR, _cid), subset in plot_data.groupby(["DL_OR", "cid"]):
        trace_name = f"{_OR}_{_cid}"  # Unique name for each trace
        fig.add_trace(
            go.Scatter(
                x=subset['PCA_1'],
                y=subset['PCA_2'],
                name=trace_name,
                marker=dict(
                    color=colormap[_OR], 
                    size=15 if subset.FDR.values[0] < 0.05 else 5, 
                    symbol=shape_dict[_cid], 
                    opacity=0.7
                ),
                hovertemplate=f'{_OR}<br>{subset.odor.values[0]}<br>logFC: {subset.logFC.values[0]:.2f}<br>FDR: {subset.FDR.values[0]:.2e}',
                mode='markers',
                legendgroup=_OR,
                # Only show in legend for first subplot to avoid duplicates
                showlegend=(row_idx == 1 and col_idx == 1)
            ),
            row=row_idx, col=col_idx
        )
    
    # Update axes labels for each subplot
    fig.update_xaxes(title_text=f"PCA_1 ({100*results[_method]['explained_variance'][0]:.3f}%)", row=row_idx, col=col_idx)
    fig.update_yaxes(title_text=f"PCA_2 ({100*results[_method]['explained_variance'][1]:.3f}%)", row=row_idx, col=col_idx)
    
    # Move to next subplot position
    col_idx += 1
    if col_idx > 2:
        col_idx = 1
        row_idx += 1

# Update layout for the entire figure
fig.update_layout(
    # height=800,  # Adjust height as needed
    # width=1000,  # Adjust width as needed
    template='simple_white',
    title_text="PCA Plots AF3 structure encoding method Comparison",
    legend_title_text="DL_OR_cid"
)

fig.show()
# fig.write_html('/mnt/data2/Justice/OR_learning/output/Canonical_bc/AF3/pS6_screen/PCA_encoding_method_cav_res.html')
fig.write_html('/mnt/data2/Justice/OR_learning/output/Canonical_bc/AF3/pS6_screen/PCA_encoding_method_res.html')

In [4]:
import numpy as np
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from scipy.ndimage import uniform_filter

def create_3d_kmers(voxel_data, kmer_size=3, stride=1, use_labeled=True):
    """
    Create 3D k-mers (cubic neighborhoods) from voxel data
    
    Parameters:
    -----------
    voxel_data : 3D or 4D array
        Either labeled voxels (3D array with integers) or 
        one-hot encoded voxels (4D array with binary values)
    kmer_size : int
        Size of cubic neighborhood (k×k×k)
    stride : int
        Step size when moving the kmer window
    use_labeled : bool
        If True, assumes voxel_data is a 3D array with integer labels
        If False, assumes voxel_data is a 4D array with one-hot encoding
        
    Returns:
    --------
    features : array
        Flattened k-mer features
    """
    if use_labeled:
        # 3D array with integer labels
        x_size, y_size, z_size = voxel_data.shape
        n_properties = 8  # Assuming 0-7 property classes
    else:
        # 4D array with one-hot encoding
        x_size, y_size, z_size, n_properties = voxel_data.shape
    
    # Calculate output dimensions
    x_out = (x_size - kmer_size) // stride + 1
    y_out = (y_size - kmer_size) // stride + 1
    z_out = (z_size - kmer_size) // stride + 1
    
    # Initialize feature vector
    feature_size = x_out * y_out * z_out * (kmer_size**3) * (1 if use_labeled else n_properties)
    features = np.zeros(feature_size)
    
    idx = 0
    # Iterate over all possible kmer positions with stride
    for i in range(0, x_size - kmer_size + 1, stride):
        for j in range(0, y_size - kmer_size + 1, stride):
            for k in range(0, z_size - kmer_size + 1, stride):
                # Extract the k×k×k neighborhood
                if use_labeled:
                    kmer = voxel_data[i:i+kmer_size, j:j+kmer_size, k:k+kmer_size]
                    # Flatten the kmer
                    features[idx:idx+kmer.size] = kmer.flatten()
                    idx += kmer.size
                else:
                    kmer = voxel_data[i:i+kmer_size, j:j+kmer_size, k:k+kmer_size, :]
                    # Flatten the kmer
                    features[idx:idx+kmer.size] = kmer.flatten()
                    idx += kmer.size
    
    return features

def weighted_spatial_encoding(voxel_data, use_labeled=True, radius=2):
    """
    Create a weighted spatial encoding where each position is influenced by its neighbors
    
    Parameters:
    -----------
    voxel_data : 3D or 4D array
        Either labeled voxels (3D with integers) or one-hot encoded (4D)
    use_labeled : bool
        If True, assumes voxel_data is a 3D array with integer labels (0-7)
        If False, assumes voxel_data is a 4D array with one-hot encoding
    radius : int
        Neighborhood radius to consider for spatial weights
        
    Returns:
    --------
    encoded_data : array
        Spatially encoded features
    """
    if use_labeled:
        # Convert labeled data to one-hot encoding
        x_size, y_size, z_size = voxel_data.shape
        n_properties = 8  # Assuming 0-7 property classes
        one_hot = np.zeros((x_size, y_size, z_size, n_properties), dtype=np.float32)
        
        for i in range(x_size):
            for j in range(y_size):
                for k in range(z_size):
                    prop_idx = voxel_data[i, j, k]
                    if 0 <= prop_idx < n_properties:
                        one_hot[i, j, k, prop_idx] = 1.0
        
        data = one_hot
    else:
        data = voxel_data.copy().astype(np.float32)
    
    x_size, y_size, z_size, n_properties = data.shape
    
    # Create spatially weighted features
    weighted_features = np.zeros_like(data)
    
    # Apply spatial averaging with different weights based on distance
    for prop in range(n_properties):
        prop_data = data[:, :, :, prop]
        
        # Create multiple weighted versions with different kernels/radii
        for r in range(1, radius + 1):
            # Use uniform filter with different sizes to approximate weighted neighborhoods
            weight = 1.0 / (r * 2)
            smoothed = uniform_filter(prop_data, size=r*2-1) * weight
            weighted_features[:, :, :, prop] += smoothed
        
        # Add the original data with higher weight
        weighted_features[:, :, :, prop] += prop_data * 0.5
    
    # Flatten while preserving some spatial structure
    # We'll divide the volume into regions and compute statistics
    region_size = max(2, min(x_size, y_size, z_size) // 4)
    
    encoded_features = []
    
    # Iterate over regions
    for i in range(0, x_size, region_size):
        i_end = min(i + region_size, x_size)
        for j in range(0, y_size, region_size):
            j_end = min(j + region_size, y_size)
            for k in range(0, z_size, region_size):
                k_end = min(k + region_size, z_size)
                
                # Extract region
                region = weighted_features[i:i_end, j:j_end, k:k_end, :]
                
                if region.size > 0:
                    # Calculate statistics for each property in this region
                    for prop in range(n_properties):
                        prop_data = region[:, :, :, prop]
                        
                        # Add various statistics
                        encoded_features.append(np.mean(prop_data))
                        encoded_features.append(np.sum(prop_data))
                        encoded_features.append(np.max(prop_data))
                        
                        # Count non-zero values (presence)
                        encoded_features.append(np.count_nonzero(prop_data))
                        
                        # Add spatial gradients (changes in x, y, z directions)
                        if prop_data.shape[0] > 1:
                            encoded_features.append(np.mean(np.diff(prop_data, axis=0)))
                        else:
                            encoded_features.append(0)
                            
                        if prop_data.shape[1] > 1:
                            encoded_features.append(np.mean(np.diff(prop_data, axis=1)))
                        else:
                            encoded_features.append(0)
                            
                        if prop_data.shape[2] > 1:
                            encoded_features.append(np.mean(np.diff(prop_data, axis=2)))
                        else:
                            encoded_features.append(0)
    
    return np.array(encoded_features)

def fourier_encoding(voxel_data, use_labeled=True, freq_cutoff=3):
    """
    Use 3D Fourier transform to capture spatial patterns in voxel data
    
    Parameters:
    -----------
    voxel_data : 3D or 4D array
        Either labeled voxels (3D with integers) or one-hot encoded (4D)
    use_labeled : bool
        If True, assumes voxel_data is a 3D array with integer labels
        If False, assumes voxel_data is a 4D array with one-hot encoding
    freq_cutoff : int
        Maximum frequency to retain in the Fourier transform
        
    Returns:
    --------
    features : array
        Fourier-encoded features
    """
    if use_labeled:
        # Convert labeled data to one-hot encoding
        x_size, y_size, z_size = voxel_data.shape
        n_properties = 8  # Assuming 0-7 property classes
        one_hot = np.zeros((x_size, y_size, z_size, n_properties), dtype=np.float32)
        
        for i in range(x_size):
            for j in range(y_size):
                for k in range(z_size):
                    prop_idx = voxel_data[i, j, k]
                    if 0 <= prop_idx < n_properties:
                        one_hot[i, j, k, prop_idx] = 1.0
        
        data = one_hot
    else:
        data = voxel_data.copy().astype(np.float32)
    
    x_size, y_size, z_size, n_properties = data.shape
    
    features = []
    
    # Apply 3D FFT to each property channel
    for prop in range(n_properties):
        prop_data = data[:, :, :, prop]
        
        # Compute 3D FFT
        fft_data = np.fft.fftn(prop_data)
        
        # Extract magnitudes of low frequency components
        # These capture the overall spatial patterns
        for i in range(-freq_cutoff, freq_cutoff + 1):
            for j in range(-freq_cutoff, freq_cutoff + 1):
                for k in range(-freq_cutoff, freq_cutoff + 1):
                    if abs(i) + abs(j) + abs(k) <= freq_cutoff:  # Focus on lower frequencies
                        # Get FFT magnitude at this frequency
                        freq_i = i % x_size
                        freq_j = j % y_size
                        freq_k = k % z_size
                        features.append(np.abs(fft_data[freq_i, freq_j, freq_k]))
    
    return np.array(features)

def spherical_harmonic_encoding(voxel_data, use_labeled=True, max_r=5, max_l=3):
    """
    Simple approximation of spherical harmonic encoding for voxel data
    
    Parameters:
    -----------
    voxel_data : 3D or 4D array
        Either labeled voxels (3D with integers) or one-hot encoded (4D)
    use_labeled : bool
        If True, assumes voxel_data is a 3D array with integer labels
        If False, assumes voxel_data is a 4D array with one-hot encoding
    max_r : int
        Maximum radius to consider for shells
    max_l : int
        Maximum order of spherical approximation
        
    Returns:
    --------
    features : array
        Encoded features
    """
    if use_labeled:
        # Convert labeled data to one-hot encoding
        x_size, y_size, z_size = voxel_data.shape
        n_properties = 8  # Assuming 0-7 property classes
        one_hot = np.zeros((x_size, y_size, z_size, n_properties), dtype=np.float32)
        
        for i in range(x_size):
            for j in range(y_size):
                for k in range(z_size):
                    prop_idx = voxel_data[i, j, k]
                    if 0 <= prop_idx < n_properties:
                        one_hot[i, j, k, prop_idx] = 1.0
        
        data = one_hot
    else:
        data = voxel_data.copy().astype(np.float32)
    
    x_size, y_size, z_size, n_properties = data.shape
    
    # Find the center of the voxel grid
    center_x, center_y, center_z = x_size // 2, y_size // 2, z_size // 2
    
    features = []
    
    # For each property, calculate features based on spherical shells
    for prop in range(n_properties):
        prop_data = data[:, :, :, prop]
        
        # For each radius, calculate shells
        for r in range(1, max_r + 1):
            shell_values = []
            
            # Collect values at approximate distance r from center
            for i in range(x_size):
                for j in range(y_size):
                    for k in range(z_size):
                        # Manhattan distance as approximation
                        dist = abs(i - center_x) + abs(j - center_y) + abs(k - center_z)
                        if r-1 <= dist <= r:
                            shell_values.append((prop_data[i, j, k], i, j, k))
            
            if shell_values:
                # Basic statistics for this shell
                values = [v[0] for v in shell_values]
                features.append(np.mean(values))
                features.append(np.sum(values))
                features.append(np.max(values))
                
                # Simple directional features (approximating spherical harmonics)
                for l in range(1, max_l + 1):
                    # For each order l, compute approximate directional components
                    if l == 1:  # First order - x, y, z directions
                        # X direction
                        x_pos = np.mean([v[0] for v in shell_values if v[1] > center_x])
                        x_neg = np.mean([v[0] for v in shell_values if v[1] < center_x])
                        features.append(x_pos - x_neg)
                        
                        # Y direction
                        y_pos = np.mean([v[0] for v in shell_values if v[2] > center_y])
                        y_neg = np.mean([v[0] for v in shell_values if v[2] < center_y])
                        features.append(y_pos - y_neg)
                        
                        # Z direction
                        z_pos = np.mean([v[0] for v in shell_values if v[3] > center_z])
                        z_neg = np.mean([v[0] for v in shell_values if v[3] < center_z])
                        features.append(z_pos - z_neg)
                    
                    elif l == 2:  # Second order - quadrants
                        # Compute averages in 8 octants
                        for x_dir in [-1, 1]:
                            for y_dir in [-1, 1]:
                                for z_dir in [-1, 1]:
                                    octant = [v[0] for v in shell_values if 
                                             (v[1] - center_x) * x_dir > 0 and
                                             (v[2] - center_y) * y_dir > 0 and
                                             (v[3] - center_z) * z_dir > 0]
                                    if octant:
                                        features.append(np.mean(octant))
                                    else:
                                        features.append(0)
                    
                    elif l == 3:  # Third order - more detailed patterns
                        # More detailed patterns could be added here
                        pass
    
    return np.array(features)

def spatial_pca_analysis(voxels, method='weighted', use_labeled=True, n_components=2):
    """
    Apply PCA to spatially encoded voxel data
    
    Parameters:
    -----------
    voxels : list of arrays
        List of voxel data (either 3D labeled or 4D one-hot encoded)
    method : str
        Encoding method: 'kmer', 'weighted', 'fourier', or 'spherical'
    use_labeled : bool
        Whether voxels are in labeled format (3D) or one-hot encoded (4D)
    n_components : int
        Number of principal components to extract
        
    Returns:
    --------
    transformed_data : array
        Data projected onto principal components
    pca : PCA object
        Fitted PCA model
    encoded_data : array
        Encoded feature vectors
    """
    # Apply selected encoding method
    encoded_data = []
    
    for voxel in voxels:
        if method == 'kmer':
            features = create_3d_kmers(voxel, kmer_size=3, stride=1, use_labeled=use_labeled)
        elif method == 'weighted':
            features = weighted_spatial_encoding(voxel, use_labeled=use_labeled, radius=2)
        elif method == 'fourier':
            features = fourier_encoding(voxel, use_labeled=use_labeled, freq_cutoff=3)
        elif method == 'spherical':
            features = spherical_harmonic_encoding(voxel, use_labeled=use_labeled, max_r=5, max_l=3)
        else:
            raise ValueError(f"Unknown method: {method}")
        
        encoded_data.append(features)
    
    # Convert to numpy array
    encoded_data = np.array(encoded_data)
    
    # Apply scaling before PCA
    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(encoded_data)
    
    # Apply PCA
    pca = PCA(n_components=n_components)
    transformed_data = pca.fit_transform(scaled_data)
    
    return transformed_data, pca, encoded_data

# Example usage with visualization
def visualize_pca_results(transformed_data, labels=None, title="Spatial PCA Analysis"):
    """Visualize the PCA results with colored points by label if provided"""
    plt.figure(figsize=(10, 8))
    
    if labels is not None:
        unique_labels = np.unique(labels)
        colors = plt.cm.tab10(np.linspace(0, 1, len(unique_labels)))
        
        for i, label in enumerate(unique_labels):
            mask = labels == label
            plt.scatter(
                transformed_data[mask, 0],
                transformed_data[mask, 1],
                c=[colors[i]],
                label=f"Class {label}",
                alpha=0.7
            )
        plt.legend()
    else:
        plt.scatter(transformed_data[:, 0], transformed_data[:, 1], alpha=0.7)
        
    plt.title(title)
    plt.xlabel('Principal Component 1')
    plt.ylabel('Principal Component 2')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.tight_layout()
    
    return plt

# Example comparing methods
def compare_encoding_methods(voxels, labels=None, use_labeled=True):
    """Compare different spatial encoding methods"""
    methods = ['kmer', 'weighted', 'fourier', 'spherical']
    results = {}
    
    plt.figure(figsize=(16, 12))
    
    for i, method in enumerate(methods):
        print(f"Applying {method} encoding...")
        transformed_data, pca, _ = spatial_pca_analysis(
            voxels, method=method, use_labeled=use_labeled, n_components=2
        )
        
        results[method] = {
            'transformed_data': transformed_data,
            'explained_variance': pca.explained_variance_ratio_
        }
        
        plt.subplot(2, 2, i+1)
        if labels is not None:
            unique_labels = np.unique(labels)
            colors = plt.cm.tab10(np.linspace(0, 1, len(unique_labels)))
            
            for j, label in enumerate(unique_labels):
                mask = labels == label
                plt.scatter(
                    transformed_data[mask, 0],
                    transformed_data[mask, 1],
                    c=[colors[j]],
                    label=f"Class {label}",
                    alpha=0.7
                )
            if i == 0:  # Only show legend once
                plt.legend()
        else:
            plt.scatter(transformed_data[:, 0], transformed_data[:, 1], alpha=0.7)
            
        explained_var = pca.explained_variance_ratio_
        plt.title(f"{method.capitalize()} Encoding\nVar: {explained_var[0]:.2f}, {explained_var[1]:.2f}")
        plt.xlabel('PC1')
        plt.ylabel('PC2')
        plt.grid(True, linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    plt.savefig('encoding_methods_comparison.png')
    plt.close()
    
    return results

### ligand RMSD vs Grantham Distance

#TODO For a given ligand, how are the different OR binding to the ligand. One can use RMSD to compare the ligand position. Then correlate the ligand RMSD with the residue something ? 

In [15]:
"""
Read in pdb files and extract only ligand coordinates 
"""

# get all pdb file paths in pS6_screen 
base_dir = "/mnt/data2/Justice/AF3_files/AF3_out/pS6_screen/"
pdb_files = [os.path.join(base_dir, sub_dir, f"{sub_dir}_model.pdb") for sub_dir in os.listdir(base_dir)] 

# Filter pdb_files by only the ones with FDR < 0.05
pS6_OR = pd.read_csv('/mnt/data2/Justice/OR_learning/output/Canonical_bc/AF3/pS6_screen/or_screen.csv', index_col=0)
pos_or_cid = list(pS6_OR[pS6_OR.FDR < 0.05].or_cid.values)
pdb_files = [_pdb for _pdb in pdb_files if "_".join([os.path.basename(_pdb).split('_')[i] for i in (0, 2)]) in pos_or_cid]


ligand_coords = {}
for _pdb in pdb_files: 
    _or_cid = "_".join([os.path.basename(_pdb).split('_')[i] for i in (0, 2)])
    ligand_coords[_or_cid] = pu.load_pdb_coordinates(_pdb, keep_ligand=True)[3]
    

In [ ]:
"""
Visualize ligand in 3D Space 
"""
fig = pf.plot_coordinates([ligand_coords[_ligand] for _ligand in ligand_coords], 
                    labels = [_ligand for _ligand in ligand_coords])

fig_backbone = pf.plot_coordinates([pu.load_pdb_coordinates('/mnt/data2/Justice/AF_files/AF_tmaligned_pdb/Olfr1377_tmaligned.pdb')[1]], 
                          labels=['backbone'], 
                          colors='#D3D3D3', 
                          opacity=0.5, 
                          size=5,
                          mode='markers')

fig.add_traces(fig_backbone.data)

fig.update_layout(title_text = 'Ligands of AF3 OR-ligand pairs<br>only showing pS6-IP FDR < 0.05', 
                  margin=dict(t=50))
fig.show()
# fig.write_html('/mnt/data2/Justice/OR_learning/output/Canonical_bc/AF3/pS6_screen/ligand_comparison/Demo_ligand_overlap.html')

In [17]:
import itertools

def rmsd(coords1, coords2):
    """Compute the RMSD between two sets of coordinates."""
    return np.sqrt(np.mean(np.sum((coords1 - coords2) ** 2, axis=1)))

def pairwise_rmsd(coord_dict):
    """Compute pairwise RMSD between all coordinate sets in the dictionary."""
    rmsd_results = {}
    for (key1, coords1), (key2, coords2) in itertools.combinations(coord_dict.items(), 2):
        rmsd_value = rmsd(coords1, coords2)
        rmsd_results[(key1, key2)] = rmsd_value
    return rmsd_results

In [25]:
"""
Conduct pairwise ligand-RMSD, PCA euclidean distance 

Make ligand_comparison_df
"""

# Conduct pairwise ligand-RMSD 
rmsd_df = pd.DataFrame()
ligand_keys = np.unique([ _key.split('_')[1] for _key in ligand_coords.keys()])
for _ligand in ligand_keys: 
    ligand_data =  { _key :ligand_coords[_key] for _key in ligand_coords if _ligand in _key}

    ligand_rmsd = pairwise_rmsd(ligand_data)
    ligand_rmsd_df = pd.DataFrame([(*_pair, _pair, _rmsd) for _pair, _rmsd in ligand_rmsd.items()], 
                              columns=["OR_1", "OR_2", "or_cid_pair","ligand_RMSD"])
    ligand_rmsd_df['cid'] = _ligand
    
    rmsd_df = pd.concat([rmsd_df, ligand_rmsd_df])

# For merging purpose 
rmsd_df['OR_1'] = rmsd_df['OR_1'].str.split('_').str[0]
rmsd_df['OR_2'] = rmsd_df['OR_2'].str.split('_').str[0]
rmsd_df['OR_pair'] = rmsd_df['OR_1'] + '_' + rmsd_df['OR_2']


# Compute PCA Euclidean distance
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

Cbc_cav_coords  = pd.read_pickle('/mnt/data2/Justice/OR_learning/files/pS6_screen/dict_pS6_screen_CBC_cav_coords.pkl')
Cbc_res_coords  = pd.read_pickle('/mnt/data2/Justice/OR_learning/files/pS6_screen/dict_pS6_screen_CBC_res_coords.pkl')
# Filter pdb_files by only the ones with FDR < 0.05
pS6_OR = pd.read_csv('/mnt/data2/Justice/OR_learning/output/Canonical_bc/AF3/pS6_screen/or_screen.csv', index_col=0)
pos_or_cid = list(pS6_OR[pS6_OR.FDR < 0.05].or_cid.values)
# PCA FOR ONLY FDR < 0.05
pca_df, variance_ratio = vf.coords_to_voxel_pca(cavity_coords  = {_or_cid: Cbc_cav_coords[_or_cid] for _or_cid in pos_or_cid},
                                                residue_coords = {_or_cid: Cbc_res_coords[_or_cid] for _or_cid in pos_or_cid},
                                                resolution = 1 )
pca_df['cid'] = pca_df['or_cid'].apply(lambda x : x.split('_')[1])

from scipy.spatial.distance import euclidean
# Conduct pairwise PCA euclidean distance 
pairwise_distances = []
for cid, group in pca_df.groupby("cid"):
    or_cids = group["or_cid"].values
    pca_coords = group[["PCA_1", "PCA_2"]].values  # Extract PCA coordinates
    # Compute pairwise Euclidean distances
    for (i, or1), (j, or2) in itertools.combinations(enumerate(or_cids), 2):
        dist = euclidean(pca_coords[i], pca_coords[j])
        pairwise_distances.append([cid, or1, or2, dist])
pairwise_df = pd.DataFrame(pairwise_distances, columns=["cid", "OR_1", "OR_2", "Euclidean_Distance"])
pairwise_df['OR_pair'] = pairwise_df['OR_1'].str.split('_').str[0].str.lower() + '_' + pairwise_df['OR_2'].str.split('_').str[0].str.lower()

ligand_comparison_df = pd.merge(rmsd_df, pairwise_df[['OR_pair', 'Euclidean_Distance']], on='OR_pair')


# Prepare grantham distance df 
grantham_df = pd.read_csv('/mnt/data2/Justice/OR_learning/output/Canonical_bc/SeqAlignment/SeqAlignment_OBR_grantham_dist_mtx.csv', index_col = 0)
grantham_df = grantham_df.reset_index(names = 'OR_1').melt(id_vars = 'OR_1', 
                                                           var_name = 'OR_2', 
                                                           value_name = 'Grantham_distance')
grantham_df = grantham_df[grantham_df['OR_1'].str.startswith('Or') & 
                          grantham_df['OR_2'].str.startswith('Or')]
grantham_df['OR_pair'] = grantham_df['OR_1'].str.lower() + '_' + grantham_df['OR_2'].str.lower()

# Merge Grantham distance with rmsd df 
ligand_comparison_df = pd.merge(ligand_comparison_df, grantham_df[['OR_pair', 'Grantham_distance']], on='OR_pair')


# Merge odor string to rmsd df
cid_odor = pd.read_csv('/mnt/data2/Justice/OR_learning/output/Canonical_bc/AF3/pS6_screen/or_screen.csv', index_col=0)[['cid', 'odor']]
cid_odor = cid_odor.drop_duplicates().reset_index(drop=True)
cid_odor['cid'] = cid_odor.cid.astype(str)
ligand_comparison_df = pd.merge(ligand_comparison_df, cid_odor, on='cid')

Original features: 13520, Reduced features: 1805
Reduced data shape: (35, 2)
Explained variance ratio: [0.09351728 0.07846383]


#### CBC PCA Euclidean distance correlation

In [ ]:
# Manually filter out for 'or4b1d_1049' as it is not binded within the binding cavity. . . 
plot_df = ligand_comparison_df[ligand_comparison_df["or_cid_pair"].apply(lambda x: 'or4b1d_1049' not in x)]

fig = pf.plot_correlation(plot_df, 
                          x_by = 'ligand_RMSD', y_by = 'Euclidean_Distance', 
                          title="Pairwise AF3 Ligand RMSD vs Euclidean Distance (CBC, cav_res)",
                          color_by = 'odor', label_by = 'OR_pair', 
                          xlabel = 'Pairwise AF3 ligand-RMSD', 
                          ylabel = 'Euclidean distance (CBC, cav_res)', 
                          linestyle='dash', 
                          figsize=None                      
                          )
fig.show()
# fig.write_html('/mnt/data2/Justice/OR_learning/output/Canonical_bc/AF3/pS6_screen/ligand_comparison/ligRMSD_EuclDistCBCcav_res.html')

In [256]:
from plotly.subplots import make_subplots

# Filter out 'or4b1d_1049'
plot_df = ligand_comparison_df[ligand_comparison_df["or_cid_pair"].apply(lambda x: 'or4b1d_1049' not in x)]

# Get unique 'odors' values
odors = plot_df["odor"].unique()
num_cids = len(odors)

# Determine subplot grid size (aiming for square-like layout)
cols = int(np.ceil(np.sqrt(num_cids)))  # Square root for balanced grid
rows = int(np.ceil(num_cids / cols))

# Create subplot layout
fig = make_subplots(rows=rows, cols=cols, subplot_titles=[f"{_odor}" for _odor in odors])

# Loop through each 'cid' and create a subplot
for i, _odor in enumerate(odors):
    row, col = divmod(i, cols)  # Get row and col position

    # Subset data for this 'cid'
    subset_df = plot_df[plot_df["odor"] == _odor]

    # Generate individual plot using your function
    sub_fig = pf.plot_correlation(subset_df, 
                                  x_by='ligand_RMSD', y_by='Euclidean_Distance', 
                                  label_by='OR_pair', 
                                  linestyle='dash')

    # Add traces to subplot
    for _trace in sub_fig.data:
        fig.add_trace(_trace, row=row+1, col=col+1) 
    
    # Extract Pearson correlation annotation text
    if sub_fig.layout.annotations:
        fig.add_annotation(
            text=sub_fig.layout.annotations[0].text,
            x=1.8, y=45,  # Keep it at upper-left of each subplot
            showarrow=False,
            opacity=0.8, 
            row=row+1, col=col+1
        )

# Apply fixed x/y limits to all subplots
x = [_x for _data in fig.data for _x in _data['x']]
y = [_y for _data in fig.data for _y in _data['y']]
padding_x = (np.max(x) - np.min(x)) * 0.05  # 5% padding
padding_y = (np.max(y) - np.min(y)) * 0.05
fig.update_xaxes(range=[np.min(x) - padding_x, np.max(x) + padding_x], title_text="ligand-RMSD")
fig.update_yaxes(range=[np.min(y) - padding_y, np.max(y) + padding_y], title_text="Euclidean Distance")

# Update layout
fig.update_layout(
    title="Pairwise AF3 Ligand RMSD vs Euclidean Distance (CBC, cav_res) for Each Odor",
    height=350 * rows,  # Adjust based on rows
    width=350 * cols,   # Adjust based on cols
    showlegend=False, 
    template="simple_white"
)

# Show figure
fig.show()
fig.write_html('/mnt/data2/Justice/OR_learning/output/Canonical_bc/AF3/pS6_screen/ligand_comparison/ligRMSD_EuclDistCBCcav_res_subplot.html')

#### Grantham Distance Correlation

In [26]:
# Manually filter out for 'or4b1d_1049' as it is not binded within the binding cavity. . . 
plot_df = ligand_comparison_df[ligand_comparison_df["or_cid_pair"].apply(lambda x: 'or4b1d_1049' not in x)]

fig = pf.plot_correlation(plot_df, 
                          x_by = 'ligand_RMSD', y_by = 'Grantham_distance', 
                          title="Pairwise AF3 Ligand RMSD vs Grantham Distance (5A OBR)",
                          color_by = 'odor', label_by = 'OR_pair', 
                          xlabel = 'Pairwise AF3 ligand-RMSD', 
                          ylabel = 'Grantham Distance (Full Sequence)', 
                          linestyle='dash', 
                          figsize=None                      
                          )
fig.show()
# fig.write_html('/mnt/data2/Justice/OR_learning/output/Canonical_bc/AF3/pS6_screen/ligand_comparison/ligRMSD_GranDistfull.html')
fig.write_html('/mnt/data2/Justice/OR_learning/output/Canonical_bc/AF3/pS6_screen/ligand_comparison/ligRMSD_GranDistOBR5A.html')

In [ ]:
from plotly.subplots import make_subplots

# Filter out 'or4b1d_1049'
plot_df = ligand_comparison_df[ligand_comparison_df["or_cid_pair"].apply(lambda x: 'or4b1d_1049' not in x)]

# Get unique 'odors' values
odors = plot_df["odor"].unique()
num_cids = len(odors)

# Determine subplot grid size (aiming for square-like layout)
cols = int(np.ceil(np.sqrt(num_cids)))  # Square root for balanced grid
rows = int(np.ceil(num_cids / cols))

# Create subplot layout
fig = make_subplots(rows=rows, cols=cols, subplot_titles=[f"{_odor}" for _odor in odors])

# Loop through each 'cid' and create a subplot
for i, _odor in enumerate(odors):
    row, col = divmod(i, cols)  # Get row and col position

    # Subset data for this 'cid'
    subset_df = plot_df[plot_df["odor"] == _odor]

    # Generate individual plot using your function
    sub_fig = pf.plot_correlation(subset_df, 
                                  x_by='ligand_RMSD', y_by='Grantham_distance', 
                                  label_by='OR_pair', 
                                  linestyle='dash')

    # Add traces to subplot
    for _trace in sub_fig.data:
        fig.add_trace(_trace, row=row+1, col=col+1) 
    
    # Extract Pearson correlation annotation text
    if sub_fig.layout.annotations:
        fig.add_annotation(
            text=sub_fig.layout.annotations[0].text,
            x=4.5, y=13,  # Keep it at upper-left of each subplot
            showarrow=False,
            opacity=0.8, 
            row=row+1, col=col+1
        )

# Apply fixed x/y limits to all subplots
x = [_x for _data in fig.data for _x in _data['x']]
y = [_y for _data in fig.data for _y in _data['y']]
padding_x = (np.max(x) - np.min(x)) * 0.05  # 5% padding
padding_y = (np.max(y) - np.min(y)) * 0.05
fig.update_xaxes(range=[np.min(x) - padding_x, np.max(x) + padding_x], title_text="ligand-RMSD")
fig.update_yaxes(range=[np.min(y) - padding_y, np.max(y) + padding_y], title_text="Grantham Distance")

# Update layout
fig.update_layout(
    # title="Pairwise AF3 Ligand RMSD vs Grantham Distance (Full Sequence) for Each Odor",
    title="Pairwise AF3 Ligand RMSD vs Grantham Distance (5A OBR) for Each Odor",
    height=350 * rows,  # Adjust based on rows
    width=350 * cols,   # Adjust based on cols
    showlegend=False, 
    template="simple_white"
)

# Show figure
fig.show()
# fig.write_html('/mnt/data2/Justice/OR_learning/output/Canonical_bc/AF3/pS6_screen/ligand_comparison/ligRMSD_GranDistfull_subplot.html')
fig.write_html('/mnt/data2/Justice/OR_learning/output/Canonical_bc/AF3/pS6_screen/ligand_comparison/ligRMSD_GranDistOBR5A_subplot.html')

### Ligand RMSD vs CBC PCA 

In [212]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

Cbc_cav_coords  = pd.read_pickle('/mnt/data2/Justice/OR_learning/files/pS6_screen/dict_pS6_screen_CBC_cav_coords.pkl')
Cbc_surf_coords = pd.read_pickle('/mnt/data2/Justice/OR_learning/files/pS6_screen/dict_pS6_screen_CBC_surf_coords.pkl')
Cbc_res_coords  = pd.read_pickle('/mnt/data2/Justice/OR_learning/files/pS6_screen/dict_pS6_screen_CBC_res_coords.pkl')


# Filter pdb_files by only the ones with FDR < 0.05
pS6_OR = pd.read_csv('/mnt/data2/Justice/OR_learning/output/Canonical_bc/AF3/pS6_screen/or_screen.csv', index_col=0)
pos_or_cid = list(pS6_OR[pS6_OR.FDR < 0.05].or_cid.values)

# PCA FOR ONLY FDR < 0.05
pca_df, variance_ratio = vf.coords_to_voxel_pca(cavity_coords  = {_or_cid: Cbc_cav_coords[_or_cid] for _or_cid in pos_or_cid},
                                                residue_coords = {_or_cid: Cbc_res_coords[_or_cid] for _or_cid in pos_or_cid},
                                                resolution = 1 )

Original features: 13520, Reduced features: 1805
Reduced data shape: (35, 2)
Explained variance ratio: [0.09436403 0.0783545 ]


In [ ]:
from itertools import combinations
from scipy.spatial.distance import euclidean

# Conduct pairwise PCA euclidean distance 
pairwise_distances = []

# Group data by 'cid'
for cid, group in pca_df.groupby("cid"):
    or_cids = group["or_cid"].values
    pca_coords = group[["PCA_1", "PCA_2"]].values  # Extract PCA coordinates

    # Compute pairwise Euclidean distances
    for (i, or1), (j, or2) in combinations(enumerate(or_cids), 2):
        dist = euclidean(pca_coords[i], pca_coords[j])
        pairwise_distances.append([cid, or1, or2, dist])

# Convert results to a DataFrame
pairwise_df = pd.DataFrame(pairwise_distances, columns=["cid", "OR_1", "OR_2", "Euclidean_Distance"])
pairwise_df['OR_pair'] = pairwise_df['OR_1'].str.split('_').str[0].str.lower() + '_' + pairwise_df['OR_2'].str.split('_').str[0].str.lower()

pairwise_df[['OR_pair', 'Euclidean_Distance']]

In [258]:
import importlib 
importlib.reload(pu)
importlib.reload(pf)
importlib.reload(bc)
importlib.reload(sa)
importlib.reload(vf)


<module 'voxel_functions' from '/mnt/data2/Justice/OR_learning/utils/voxel_functions.py'>